## **Imports**

In [5]:
import json
import matplotlib.pyplot as plt
import numpy as np
import re

## **Functions**

In [ ]:
def load_data(filename):
    """Loads JSON data from a specified file."""
    try:
        with open(filename, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Error: File not found: {filename}. Please ensure the JSON files are in the same directory.")
        return None
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON format in {filename}.")
        return None
    
def plot_test_score_over_epochs(data, label=None, xytext_max=(0,0), xytext_min=(0,0)):

    epochs = np.array(data["num_epochs"])
    scores = np.array(data["test_mean_score_over_epochs"])

    # Global maximum
    max_idx = np.argmax(scores)
    max_epoch, max_score = epochs[max_idx], scores[max_idx]

    # Minimum after max
    after_scores = scores[max_idx+1:]
    after_epochs = epochs[max_idx+1:]

    min_after_idx = np.argmin(after_scores)
    min_after_epoch = after_epochs[min_after_idx]
    min_after_score = after_scores[min_after_idx]

    ax = plt.gca()

    ax.plot(epochs, scores, marker='o', linewidth=2, label=label)

    # Max (green)
    ax.scatter(max_epoch, max_score, color="green", s=120, zorder=5)
    ax.annotate(
        f"Max {max_score:.2f}",
        (max_epoch, max_score),
        xytext=xytext_max,
        textcoords="offset points",
        color="green"
    )

    # Min after max (red)
    ax.scatter(min_after_epoch, min_after_score, color="red", s=120, zorder=5)
    ax.annotate(
        f"MinAfter {min_after_score:.2f}",
        (min_after_epoch, min_after_score),
        xytext=xytext_min,
        textcoords="offset points",
        color="red"
    )

def plot_losses_over_epochs(data, label=None, xytext_noise=(0,0), xytext_nll=(0,0)):

    epochs = np.array(data["num_epochs"])
    noise_loss = np.array(data["test_noise_pred_loss_epochs"])
    nll_loss = np.array(data["nll_test_over_epochs"])

    # minimum noise loss
    noise_min_idx = np.argmin(noise_loss)
    noise_min_epoch = epochs[noise_min_idx]
    noise_min_val = noise_loss[noise_min_idx]

    # minimum NLL
    nll_min_idx = np.argmin(nll_loss)
    nll_min_epoch = epochs[nll_min_idx]
    nll_min_val = nll_loss[nll_min_idx]

    # get current axes
    ax_noise = plt.subplot(2,1,1)
    ax_nll = plt.subplot(2,1,2)

    # ----- noise loss plot -----
    ax_noise.plot(epochs, noise_loss, linewidth=2, label=label)

    ax_noise.scatter(noise_min_epoch, noise_min_val, color="red", s=120, zorder=5)
    ax_noise.annotate(
        f"Min {noise_min_val:.4f}",
        (noise_min_epoch, noise_min_val),
        xytext=xytext_noise,
        textcoords="offset points",
        color="red"
    )

    # ----- nll plot -----
    ax_nll.plot(epochs, nll_loss, linewidth=2, label=label)

    ax_nll.scatter(nll_min_epoch, nll_min_val, color="red", s=120, zorder=5)
    ax_nll.annotate(
        f"Min {nll_min_val:.4f}",
        (nll_min_epoch, nll_min_val),
        xytext=xytext_nll,
        textcoords="offset points",
        color="red"
    )


def refine(data_file):
    with open(data_file, "r") as f:
        data = json.load(f)

    # collect (epoch, success_rate)
    pairs = []
    for k, v in data.items():
        if k.startswith("model_at_epoch_"):
            epoch = int(re.findall(r'\d+', k)[0])
            pairs.append((epoch, v["success_rate"]))

    # sort by epoch
    pairs.sort(key=lambda x: x[0])

    # extract success rates
    success_rates = [v for _, v in pairs]

    # replace the array
    data["test_mean_score_over_epochs"] = success_rates

    print("Updated test_mean_score_over_epochs:")
    print(data["test_mean_score_over_epochs"])
    return data


## **Load data**

In [11]:
epoch = 4500
DP_path = f"data/outputs/2026.03.28/12.15.58_train_diffusion_unet_lowdim_tool_hang_lowdim/eval_output/epoch={epoch}/"
reconst_loss_DP = np.loadtxt(DP_path + "reconstruction_loss.csv", delimiter=",")
manifold_adh_DP = np.loadtxt(DP_path + "manifold_adherence.csv", delimiter=",")
all_pred_actions_DP = np.load(DP_path + "all_pred_actions.npy")

PDP_path = f"data/outputs/2026.03.28/12.07.54_train_prob_diffusion_unet_lowdim_tool_hang_lowdim/eval_output/epoch={epoch}/"
reconst_loss_PDP = np.loadtxt(PDP_path + "reconstruction_loss.csv", delimiter=",")
manifold_adh_PDP = np.loadtxt(PDP_path + "manifold_adherence.csv", delimiter=",")
all_pred_actions_PDP = np.load(PDP_path + "all_pred_actions.npy")

## **Plot success rate**

In [17]:
print ("mean manifold adherence for DP = ", manifold_adh_DP.mean())
print ("mean reconstruction loss for DP = ", reconst_loss_DP.mean())

print ("mean manifold adherence for PDP = ", manifold_adh_PDP.mean())
print ("mean reconstruction loss for PDP = ", reconst_loss_PDP.mean())

print (manifold_adh_DP.shape)
manifold_mean_DP = manifold_adh_DP.mean(axis=1)
manifold_mean_PDP = manifold_adh_PDP.mean(axis=1)

dis_actions = np.linalg.norm(all_pred_actions_DP - all_pred_actions_PDP, axis=(2,3))

for i in range (manifold_adh_DP.shape[1]):
    print (f"step={i}   ",f"DP_adh = {manifold_adh_DP[0][i]}    ", f"PDP_adh = {manifold_adh_PDP[0][i]}")
print (dis_actions.shape)
for i in range (dis_actions.shape[1]):
    print (f"step={i}   ",f"dis_actions = {dis_actions[1][i]}")

mean manifold adherence for DP =  0.1262829801203175
mean reconstruction loss for DP =  0.9715310995638455
mean manifold adherence for PDP =  0.4048490972131152
mean reconstruction loss for PDP =  2.1299464824660257
(50, 88)
step=0    DP_adh = 0.023262178525328636     PDP_adh = 0.05058491230010986
step=1    DP_adh = 0.012852456420660019     PDP_adh = 0.031779300421476364
step=2    DP_adh = 0.011452593840658665     PDP_adh = 0.02897082269191742
step=3    DP_adh = 0.01018589362502098     PDP_adh = 0.027269864454865456
step=4    DP_adh = 0.009074377827346325     PDP_adh = 0.032131582498550415
step=5    DP_adh = 0.03408151492476463     PDP_adh = 0.06779278069734573
step=6    DP_adh = 0.03768084943294525     PDP_adh = 0.06688519567251205
step=7    DP_adh = 0.03449917957186699     PDP_adh = 0.06880581378936768
step=8    DP_adh = 0.022637654095888138     PDP_adh = 0.04952399432659149
step=9    DP_adh = 0.01717044785618782     PDP_adh = 0.02830605022609234
step=10    DP_adh = 0.019354248419404

## **Plot noise prediction error and lower bound of NLL**

In [ ]:
plt.figure(figsize=(10,8))

plot_losses_over_epochs(data1, label="DP", xytext_noise=(-30,10), xytext_nll=(-30,10))
plot_losses_over_epochs(data2, label="Ours", xytext_noise=(10,-20), xytext_nll=(10,-20))

plt.subplot(2,1,1)
plt.ylabel("Noise Prediction Loss")
plt.title("Noise Prediction Loss Comparison")
plt.grid(True)
plt.legend()

plt.subplot(2,1,2)
plt.xlabel("Epoch")
plt.ylabel("NLL")
plt.title("NLL Comparison")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig("loss_comparison_transport.png")